# Dataset cleaning and Bikeable perceived-safety scores

Addresses the dataset consistency review: reduces the collision file to
strictly pre-crash inputs plus the severity outcome, applies the five
specific cleaning steps raised, then runs the Bikeable model across the
collision images and adds the average and worst-case perceived-safety
scores to the cleaned file.


## Install dependencies

In [ ]:
import sys, subprocess
for pkg in ["lightgbm", "scikit-learn", "joblib", "azure-storage-blob", "Pillow"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)
print("Dependencies checked.")


## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT_FOLDER = r"..."
FULL_CSV = os.path.join(INPUT_FOLDER, "collisions_final.csv")
CLEAN_OUTPUT_CSV = os.path.join(INPUT_FOLDER, "collisions_precrash_clean.csv")

full = pd.read_csv(FULL_CSV, low_memory=False)
print(f"Full dataset: {full.shape[0]:,} rows, {full.shape[1]} columns")


## Part 1: the comprehensive diagnostic, every column as it really is

Before any decision gets made about what belongs in the model, this
looks at every single one of the 463 real columns exactly as it is,
its real data type, how much is really missing, its real range or its
real categories, and a few real example values. No name-based guessing
yet, just the facts. Everything from here on, what gets excluded, what
gets kept, what gets flagged, is decided using what this section
actually finds, not assumed ahead of it.


In [ ]:
def profile_raw_column(df_, col):
    series = df_[col]
    n = len(series)
    n_missing = series.isna().sum()
    non_null = series.dropna()
    profile = {
        "column": col,
        "data_type": "Number" if pd.api.types.is_numeric_dtype(series) else "Text/category",
        "missing_pct": round(n_missing / n * 100, 2),
        "distinct_count": series.nunique(dropna=True),
    }
    if pd.api.types.is_numeric_dtype(series) and len(non_null) > 0:
        profile["range_or_categories"] = f"{non_null.min()} to {non_null.max()}"
    elif len(non_null) > 0:
        if profile["distinct_count"] <= 10:
            profile["range_or_categories"] = ", ".join(str(v) for v in sorted(non_null.unique(), key=str))
        else:
            profile["range_or_categories"] = f"{profile['distinct_count']} different values, too many to list"
    else:
        profile["range_or_categories"] = "entirely empty"
    profile["examples"] = ", ".join(str(v) for v in non_null.sample(min(3, len(non_null)), random_state=1)) if len(non_null) > 0 else ""
    return profile

raw_profile = pd.DataFrame([profile_raw_column(full, col) for col in full.columns])

print(f"Profiled all {len(raw_profile)} real columns, exactly as they are in your file.")
print(f"  Numbers: {(raw_profile['data_type'] == 'Number').sum()}")
print(f"  Text/category: {(raw_profile['data_type'] == 'Text/category').sum()}")
print(f"  Completely empty: {(raw_profile['range_or_categories'] == 'entirely empty').sum()}")
print()
print("A sample of what this actually looks like, 10 real columns:")
print()
for _, row in raw_profile.sample(10, random_state=2).iterrows():
    print(f"  {row['column']} ({row['data_type']}, {row['missing_pct']}% missing)")
    print(f"    {row['range_or_categories']}")


### Using that same diagnostic to catch a specific, real risk

The same real values just profiled above make it possible to check
something directly: could a column like `age` actually describe a
specific person rather than a place. Every real column name gets
checked against a list of words that usually mean exactly that, and
the real values behind any match are shown here, so the decision made
in the next part is based on real evidence, not a name alone.


In [ ]:
PERSON_ATTRIBUTE_KEYWORDS = {"age", "sex", "gender", "height", "weight", "dob", "birth", "name", "ethnicity"}
KNOWN_GLUED_EXCEPTIONS = {"driversex", "passengersex", "casualtysex"}

def looks_like_person_attribute(col_name):
    lowered = col_name.lower()
    if lowered in KNOWN_GLUED_EXCEPTIONS:
        return True
    tokens = lowered.split("_")
    return any(t in PERSON_ATTRIBUTE_KEYWORDS for t in tokens)

flagged_cols = [c for c in full.columns if looks_like_person_attribute(c)]

print(f"Real column names matching a person-attribute word: {len(flagged_cols)}")
print()
for col in flagged_cols:
    real_values = full[col].dropna()
    print(f"{col!r}:")
    if pd.api.types.is_numeric_dtype(full[col]) and len(real_values) > 0:
        print(f"  Real range: {real_values.min()} to {real_values.max()}")
    print(f"  A few real values: {sorted(real_values.sample(min(5, len(real_values)), random_state=1).tolist(), key=str)}")
    print()

print("Each one of these needs a real answer, not a guess, about what it actually records")
print("before it goes anywhere near the kept feature list below. If any of them describe a")
print("specific person, the casualty or the driver, rather than the location, they belong")
print("with the other excluded, post-crash columns, not the kept ones.")

print()
print("=" * 70)
print("EVERY OTHER TEXT OR CATEGORY COLUMN, NOT CAUGHT BY A KEYWORD")
print("=" * 70)
print("A keyword match only catches what it is looking for. This covers the rest,")
print("every remaining text or category column, so nothing sits unexamined just")
print("because its name did not happen to contain one of the words above.")
print()

text_cols = raw_profile[raw_profile["data_type"] == "Text/category"]["column"].tolist()
remaining_text_cols = [c for c in text_cols if c not in flagged_cols]

print(f"Remaining text/category columns to check: {len(remaining_text_cols)}")
print()
for col in remaining_text_cols:
    real_values = full[col].dropna()
    n_distinct = real_values.nunique()
    if n_distinct <= 12:
        print(f"{col!r}: {sorted(real_values.unique().tolist(), key=str)}")
    else:
        print(f"{col!r}: {n_distinct} different real values, too many to list, a few examples: "
              f"{sorted(real_values.sample(min(5, len(real_values)), random_state=1).tolist(), key=str)}")



## Part 2: deciding what's genuinely pre-crash, using Part 1's diagnostic directly

Every column below is excluded for one of two reasons, both checked
directly against the real evidence from Part 1, not assumed: it
describes something only known after the crash happened, the injury,
the vehicles involved, who was hurt and how old they were, or it's an
identifier or a timestamp rather than a real predictive input.

Three of these are here specifically because of what Part 1's
diagnostic showed, not because of what the name alone suggested:
`age` showed a real range of 1 to 94, the same kind of casualty detail
as `casualty_sex` right next to it. `junctionlocation` and `cwhazard`
both describe something true only during this specific crash, where a
vehicle was sitting, what was in the road at that moment, not a fixed
property of the place beforehand.


In [ ]:
EXCLUDED_COLUMNS = {
    "Casualty and injury detail, only known after the crash": [
        "casualty_count", "casualty_injury_based", "casualty_ref", "casualty_severity",
        "casualty_sex", "casualty_adj_serious", "casualty_adj_slight", "enhanced_severity",
        "age",  # confirmed against real values: a genuine human age, 1 to 94, the same
                # kind of casualty detail as casualty_sex right above it, only known
                # once a crash has happened and someone has been examined
    ],
    "Vehicle and driver detail, only known after the crash": [
        "vehicletype", "vehiclemanoeuvres", "vehicleskidding", "vehicle_manoeuvres_simple",
        "vehimpact", "vehoffcw", "driversex", "n_other_vehicles", "other_vehicle_1",
        "other_vehicle_2", "other_vehicle_3", "objectincw", "restrictedlane",
        "journeypurpose", "noofvehiclesinacc", "noofcasualtiesinacc",
        "junctionlocation",  # where a specific vehicle was positioned during this
                             # specific crash, not a fixed property of the junction,
                             # the same kind of thing vehiclemanoeuvres already covers
        "cwhazard",  # something present in the road during this specific incident,
                     # an animal, a pedestrian, not a standing property of the location
    ],
    "Identifiers, dates and timestamps, not predictive inputs": [
        "collision_id", "UID", "dt", "time", "accidentdate", "accidentdate_parsed",
        "weather_date", "accident_time_utc",
    ],
    "Weather-matching process metadata, not real weather": [
        "archive_hour_match_gap_seconds", "archive_retrieved_latitude", "archive_retrieved_longitude",
        "archive.hourly.relativehumidity_2m", "archive.hourly.relativehumidity_2m_lag_1h",
        "archive.hourly.relativehumidity_2m_lag_3h", "archive.hourly.relativehumidity_2m_avg_last_3h",
        "archive.hourly.relativehumidity_2m_avg_last_6h",
    ],
    "Named point-of-interest identifiers, redundant once distance is kept": [
        "nearest_bus_stop_road_name", "nearest_overground_crow_name", "nearest_pub_crow_fhrsid",
        "nearest_pub_crow_name", "nearest_school_crow_name", "nearest_school_road_urn",
        "nearest_signal_crow_name", "nearest_underground_crow_name",
    ],
}

# A second category, columns excluded because a direct correlation check
# confirms they are redundant, not because their name suggests it, gets
# added further down, right after the check that actually finds them.
# Name-based reasoning alone would never have caught these.

OUTCOME_COLUMN = "casualty_severity"

all_excluded = [c for cols in EXCLUDED_COLUMNS.values() for c in cols if c != OUTCOME_COLUMN]
kept_cols = [c for c in full.columns if c not in all_excluded]

print(f"Excluded as non-pre-crash or non-predictive: {len(all_excluded)} columns")
for category, cols in EXCLUDED_COLUMNS.items():
    present = [c for c in cols if c in full.columns]
    print(f"  {category}: {len(present)}")
print()
print(f"Retained as candidate pre-crash inputs, plus the outcome: {len(kept_cols)} columns")

precrash = full[kept_cols].copy()


### Grouping every column into a family, defined once, used from here on

This grouping gets used repeatedly from this point on, in Part 3's review
just below, and again later when the final feature list is summarised by
family, so it is defined once, here, rather than twice.


In [ ]:
def categorize(col):
    if col.startswith("archive.hourly.") or col in ("accident_to_weather_distance_m", "hours_back_precipitation", "back_precipitation"):
        return "Weather"
    if col.startswith("past_accidents_") or col.startswith("prior_severe_") or col in (
        "weighted_prior_severe_memory", "hotspot_grad_1y_100m__5y_500m",
        "hotspot_grad_inv_1y_100m__5y_500m", "exposure_adjusted_collision_history"
    ):
        return "Historical collision counts and risk composites"
    if col.startswith("nearest_") or (col.startswith("n_") and any(k in col for k in
        ["100m", "pubs", "schools", "signals", "bus_stops", "overground", "underground"])):
        return "Points of interest"
    if col.startswith("IMD") or col == "HS_bin":
        return "Deprivation and high street context"
    if col.startswith("noise_"):
        return "Noise environment"
    if col.startswith("exposure_") or col in ("n_sites_in_pool", "match_level", "n_samples"):
        return "Cyclist exposure and site matching"
    if col in ("speed_energy_proxy", "bank_holiday", "grade_percent", "streetview_distance_m", "streetview_time_diff_days"):
        return "Engineered risk and calendar features"
    if col in ("road_avg_width", "road_min_width", "pavement_avg_width_m", "pavement_min_width_m",
               "streetlight_coverage", "road_route_hierarchy", "quality_flag", "snap_distance_m"):
        return "Road and street infrastructure descriptors"
    if col in ("year", "borough", "easting", "northing", "location", "day", "highway",
               "roadclass1", "roadno1", "roadtype", "speedlimit", "speedlimit_clean", "junctiondetail",
               "junctioncontrol", "roadclass2", "roadno2", "pedcrossingdecoded", "lightconditionsbanded",
               "lightconditions_collapsed", "weather", "roadsurface", "specialconditions",
               "longitude", "latitude"):
        return "Core crash record"
    return "Other / ungrouped"

FAMILY_JUSTIFICATIONS = {
    "Core crash record": "Fixed properties of the location and the conditions present at the time, speed limit, road type, junction type, lighting, weather description, road surface. These exist independent of whether a crash happens, none of them are derived from the crash's outcome.",
    "Weather": "Hourly weather readings matched to each crash by timestamp. Legitimate only if the matched reading is at or before the crash time, not after, checked directly below.",
    "Historical collision counts and risk composites": "Counts of past crashes near this location, within a fixed time and distance window. Legitimate only if that window genuinely stops before this crash, not on or after it, checked directly below.",
    "Points of interest": "Fixed physical features near the location, nearest signal, pub, school. These do not change based on whether a crash happens here.",
    "Deprivation and high street context": "Slow changing, area level statistics. Not affected by any individual crash.",
    "Noise environment": "Ambient noise level tied to the location itself, not the crash.",
    "Cyclist exposure and site matching": "How much cycling activity a location typically sees, a property of the place, not the crash.",
    "Engineered risk and calendar features": "Derived from fixed location or calendar properties, bank holiday, road grade, not from the crash's outcome.",
    "Road and street infrastructure descriptors": "Physical properties of the road itself, width, lighting, classification, fixed regardless of any crash.",
    "Other / ungrouped": "A small leftover group, reviewed individually rather than by family.",
}

print("Defined categorize() and its family justifications, used from here on, in Part 3 just below")
print("and again later when the final feature list gets summarised by family.")


## Part 3: a comprehensive, correlation-based feature review

Beyond name-based reasoning, every one of the 463 original columns gets
checked directly against real data here: does it look like an identifier,
does it ever vary, is it a near-exact duplicate of anything else, and does
it link to the real outcome more strongly than everything else does. This
is what found the redundant point-of-interest coordinates now sitting in
the excluded list above, along with a few smaller duplicates.


In [ ]:
print("=" * 70)
print("COMPREHENSIVE FEATURE REVIEW, ALL 463 ORIGINAL COLUMNS, BY FAMILY")
print("=" * 70)
print("Every one of the 463 real, original columns gets checked here, not just")
print("the 398 already kept. Four checks, run the same way for everyone:")
print("  1. Does it look like an identifier, a near-unique value with no")
print("     real generalizable meaning, the same problem 'location' had.")
print("  2. Does it never actually vary, so there is nothing for a model")
print("     to learn from it either way.")
print("  3. Is it a near-exact duplicate of ANY other column, kept or")
print("     excluded, not just a coordinate.")
print("  4. For columns not already excluded: does it correlate with the")
print("     real outcome far more strongly than everything else does.")
print("     Skipped for already-excluded columns, since a strong link to")
print("     the outcome there is expected, that is exactly why they were")
print("     excluded, not a new finding.")
print()
print("This runs against the ORIGINAL 463 columns, before Part 5's cleaning,")
print("so a few things already fixed there will show up again here too, the")
print("exposure distance triplet and the wind duplicates. That is expected,")
print("not a new problem, Part 5 already resolves both correctly.")
print()

col_to_excl_category = {}
for category, cols in EXCLUDED_COLUMNS.items():
    for c in cols:
        col_to_excl_category[c] = category

def looks_like_identifier(series, total_rows):
    distinct = series.nunique(dropna=True)
    if pd.api.types.is_numeric_dtype(series):
        return False, distinct
    return distinct / total_rows > 0.5, distinct

numeric_cols = [c for c in full.columns if c != OUTCOME_COLUMN and pd.api.types.is_numeric_dtype(full[c])]
print(f"Computing every pairwise correlation across {len(numeric_cols)} numeric columns, this takes a moment...")
full_corr = full[numeric_cols].corr()

CORE_REFERENCE_COLUMNS = {"latitude", "longitude", "easting", "northing"}

DUPLICATE_THRESHOLD = 0.99
duplicate_pairs = {}
for i, c1 in enumerate(numeric_cols):
    for c2 in numeric_cols[i + 1:]:
        v = full_corr.loc[c1, c2]
        if pd.notna(v) and abs(v) > DUPLICATE_THRESHOLD:
            if c1 in CORE_REFERENCE_COLUMNS and c2 not in CORE_REFERENCE_COLUMNS:
                duplicate_pairs.setdefault(c2, []).append((c1, v))
            elif c2 in CORE_REFERENCE_COLUMNS and c1 not in CORE_REFERENCE_COLUMNS:
                duplicate_pairs.setdefault(c1, []).append((c2, v))
            else:
                duplicate_pairs.setdefault(c1, []).append((c2, v))
                duplicate_pairs.setdefault(c2, []).append((c1, v))

# Computed for every numeric column, kept or excluded, not just kept ones.
# A post-crash column correlating strongly with the outcome is expected,
# not a warning, but showing the real number everywhere, rather than only
# where a threshold says it is worth it, means nothing has to be taken on
# trust, the full picture is there to look at directly.
severity_corr = {}
severity_binary = full[OUTCOME_COLUMN].map({"Slight": 0, "Serious": 1, "Fatal": 1})
kept_numeric_cols = [c for c in numeric_cols if c not in col_to_excl_category]
for c in numeric_cols:
    severity_corr[c] = full[c].corr(severity_binary)
# The "unusually strong" threshold below is still judged only against kept
# columns, since that is the real question: does a genuine candidate
# feature look suspiciously predictive, not whether an already-excluded
# post-crash column does, which would be expected either way.
typical_outcome_corr = pd.Series({c: severity_corr[c] for c in kept_numeric_cols}).abs().median()
OUTCOME_OUTLIER_THRESHOLD = max(0.15, typical_outcome_corr * 5)
print(f"Typical real correlation with the outcome, kept columns only: {typical_outcome_corr:.4f}")
print(f"Flagging anything above {OUTCOME_OUTLIER_THRESHOLD:.4f} as worth a second look.")
print()

review_rows = []
for col in full.columns:
    if col == OUTCOME_COLUMN:
        continue
    series = full[col]
    is_excluded = col in col_to_excl_category
    family = col_to_excl_category[col] if is_excluded else categorize(col)
    is_id_like, distinct_count = looks_like_identifier(series, len(full))
    no_variation = distinct_count <= 1

    flags = []
    if no_variation:
        flags.append("never varies, nothing to learn from it")
    if is_id_like:
        flags.append(f"looks like an identifier, {distinct_count:,} distinct values")
    if col in duplicate_pairs:
        best_match = max(duplicate_pairs[col], key=lambda x: abs(x[1]))
        flags.append(f"near-duplicate of {best_match[0]} (correlation {best_match[1]:.4f})")
    if not is_excluded and col in severity_corr and abs(severity_corr[col]) > OUTCOME_OUTLIER_THRESHOLD:
        flags.append(f"unusually strong link to the real outcome ({severity_corr[col]:+.4f})")

    suggested_action = "; ".join(flags) if flags else "KEEP"
    review_rows.append({
        "status": "Excluded" if is_excluded else "Kept",
        "family": family,
        "feature": col,
        "suggested_action": suggested_action,
    })

review_df = pd.DataFrame(review_rows)

print(f"Reviewed {len(review_df)} columns, {(review_df['status']=='Kept').sum()} kept, "
      f"{(review_df['status']=='Excluded').sum()} already excluded.")
n_flagged = (review_df["suggested_action"] != "KEEP").sum()
n_flagged_kept = ((review_df["suggested_action"] != "KEEP") & (review_df["status"] == "Kept")).sum()
n_flagged_excluded = ((review_df["suggested_action"] != "KEEP") & (review_df["status"] == "Excluded")).sum()
print(f"Flagged for a second look: {n_flagged} total ({n_flagged_kept} currently kept, {n_flagged_excluded} already excluded)")
print()

for family in sorted(review_df["family"].unique()):
    family_rows = review_df[review_df["family"] == family]
    flagged = family_rows[family_rows["suggested_action"] != "KEEP"]
    status_label = family_rows["status"].iloc[0]
    print(f"{family}  [{status_label}]  ({len(family_rows)} columns, {len(flagged)} flagged)")
    for _, row in flagged.iterrows():
        print(f"  {row['feature']}: {row['suggested_action']}")
    print()


## Part 4: acting on Part 3's findings

Every one of these was found by direct correlation evidence just above,
not by a name pattern, that is the whole point of running Part 3 in the
first place.


In [ ]:
CORRELATION_EXCLUDED = {
    "nearest_pub_crow_lat": "Correlates 1.0000 with the crash's own latitude, confirmed redundant, not a new location",
    "nearest_pub_crow_lon": "Correlates 1.0000 with the crash's own longitude, confirmed redundant, not a new location",
    "nearest_pub_road_lat": "Correlates 1.0000 with the crash's own latitude, confirmed redundant, not a new location",
    "nearest_pub_road_lon": "Correlates 1.0000 with the crash's own longitude, confirmed redundant, not a new location",
    "nearest_school_crow_lat": "Correlates 0.9996 with the crash's own latitude, confirmed redundant, not a new location",
    "nearest_school_crow_lon": "Correlates 0.9998 with the crash's own longitude, confirmed redundant, not a new location",
    "nearest_school_road_E": "Correlates 1.0000 with the crash's own easting, confirmed redundant, not a new location",
    "nearest_school_road_N": "Correlates 1.0000 with the crash's own northing, confirmed redundant, not a new location",
    "nearest_overground_crow_lat": "Correlates 0.9995 with the crash's own latitude, confirmed redundant, not a new location",
    "nearest_overground_crow_lon": "Correlates 0.9998 with the crash's own longitude, confirmed redundant, not a new location",
    "nearest_overground_road_E": "Correlates 1.0000 with another nearest-POI easting column, confirmed redundant",
    "nearest_overground_road_N": "Correlates 1.0000 with another nearest-POI northing column, confirmed redundant",
    "nearest_underground_crow_lat": "Correlates 0.9996 with the crash's own latitude, confirmed redundant, not a new location",
    "nearest_underground_crow_lon": "Correlates 0.9997 with the crash's own longitude, confirmed redundant, not a new location",
    "nearest_underground_road_E": "Correlates 1.0000 with another nearest-POI easting column, confirmed redundant",
    "nearest_underground_road_N": "Correlates 1.0000 with another nearest-POI northing column, confirmed redundant",
    "nearest_signal_crow_lat": "Correlates 0.9996 with the crash's own latitude, confirmed redundant, not a new location",
    "nearest_signal_crow_lon": "Correlates 0.9998 with the crash's own longitude, confirmed redundant, not a new location",
    "nearest_signal_road_E": "Correlates 1.0000 with the crash's own easting, confirmed redundant, not a new location",
    "nearest_signal_road_N": "Correlates 1.0000 with the crash's own northing, confirmed redundant, not a new location",
    "nearest_bus_stop_road_E": "Correlates 1.0000 with the crash's own easting, confirmed redundant, not a new location",
    "nearest_bus_stop_road_N": "Correlates 1.0000 with the crash's own northing, confirmed redundant, not a new location",
    "easting": "Correlates 0.9998 with longitude, the same physical location in a second coordinate system, "
               "longitude is kept instead since it is already used throughout the rest of this project",
    "northing": "Correlates 0.9995 with latitude, the same physical location in a second coordinate system, "
                "latitude is kept instead since it is already used throughout the rest of this project",
    "IMDRank": "Correlates 0.9922 with IMDDecile, the same deprivation measure twice, the decile is kept "
               "since it is the more commonly used form of this measure",
}

columns_before_correlation_step = precrash.shape[1]
confirmed_present = [c for c in CORRELATION_EXCLUDED if c in precrash.columns]
precrash = precrash.drop(columns=confirmed_present)

print(f"Started this step with: {columns_before_correlation_step} columns")
print(f"Removing {len(confirmed_present)} columns, each confirmed redundant by direct correlation")
print("in Part 3 above, not by name, this is the list Part 3 actually found:")
for col in confirmed_present:
    print(f"  {col}: {CORRELATION_EXCLUDED[col]}")
print()
print(f"RESULT: {len(confirmed_present)} columns removed. Now {precrash.shape[1]} columns.")


### Flagged for the team to decide, not resolved here

One pattern this turned up is not clear-cut enough to act on alone: several
4-year historical crash-count windows correlate with their neighbouring
5-year windows at 0.99, tighter than the roughly 0.95 this project treats
as the boundary for "these might be built the same way by accident". A real
accumulation effect could explain part of this, most of a location's crash
history sitting in a couple of recent years is plausible, but it is close
enough to the redundancy threshold to be worth asking directly: were the
4-year and 5-year windows for `past_accidents_300m`, `400m`, and `500m`
computed independently, or could one have been derived from the other.
Nothing has been removed for this reason, this needs an answer from
whoever built that feature set first.


## Part 5: removing columns that are empty or exact duplicates

Two different problems, both checked directly rather than assumed.


In [ ]:
# A fresh checkpoint here, not len(kept_cols) from Part 2, since Part 4
# already removed 25 more columns since then, using that older number
# here would silently undercount what Part 5 actually starts with.
columns_before_part5 = precrash.shape[1]

# Part 1's diagnostic already showed these were 100% missing, this acts on
# that finding directly rather than re-checking it from scratch.
fully_empty = raw_profile[
    (raw_profile["column"].isin(precrash.columns)) & (raw_profile["range_or_categories"] == "entirely empty")
]["column"].tolist()

print(f"Part 1 already found these {len(fully_empty)} columns were completely empty:")
for col in fully_empty:
    print(f"  {col}")

precrash = precrash.drop(columns=fully_empty)
print()
print(f"Removed. Columns remaining: {precrash.shape[1]}")
print(f"RESULT: {len(fully_empty)} columns removed from the dataset.")


### Exact duplicate distance columns

Three separate duplicate pairs or groups were raised. The coordinate
pair and the date pair are already resolved by the pre-crash filter in
Part 2, since the duplicate side of each of those was process metadata
or a timestamp, not a genuine predictor, so they never entered the
candidate set to begin with. The exposure-distance triplet is
different: all three copies are genuine candidate predictors, so this
one needs an active decision.


In [ ]:
provenance_log = []
columns_before_this_cell = precrash.shape[1]

# --- 2a. exposure distance triplet, genuinely all three still present ---
dist_cols = ["exposure_site_dist_m_x", "exposure_site_dist_m.1", "exposure_site_dist_m_y"]
dist_cols_present = [c for c in dist_cols if c in precrash.columns]
print("Exposure distance triplet, checking they are truly identical:")

all_confirmed_identical = False
if len(dist_cols_present) < 2:
    print(f"  Only found: {', '.join(dist_cols_present)}, nothing to consolidate, leaving as is.")
else:
    ref = precrash[dist_cols_present[0]]
    max_diffs = []
    for c in dist_cols_present[1:]:
        max_diff = (precrash[c] - ref).abs().max()
        max_diffs.append(max_diff)
        print(f"  Max difference between {dist_cols_present[0]} and {c}: {max_diff}")
    all_confirmed_identical = all((pd.isna(d) or d < 1e-9) for d in max_diffs)

CANONICAL_DIST_COL = "exposure_site_dist_m"
if all_confirmed_identical:
    precrash[CANONICAL_DIST_COL] = precrash[dist_cols_present[0]]
    drop_dist = dist_cols_present
    precrash = precrash.drop(columns=drop_dist)
    provenance_log.append({
        "canonical_column": CANONICAL_DIST_COL,
        "original_duplicate_columns": ", ".join(dist_cols_present),
        "reason": "All three were the same distance measurement, kept once under one clear name",
    })
    print(f"Kept as {CANONICAL_DIST_COL!r}, dropped: {', '.join(drop_dist)}")
    columns_after_step_a = precrash.shape[1]
    net_change_a = columns_after_step_a - columns_before_this_cell
    print(f"RESULT: had {columns_before_this_cell} columns, removed {len(drop_dist)} and added 1, "
          f"now {columns_after_step_a} columns (net change: {net_change_a:+d}).")
elif len(dist_cols_present) >= 2:
    print()
    print("They are NOT all exactly identical, so nothing was merged automatically.")
    print("All three columns are being kept as is, this needs a manual look before")
    print("picking one, to avoid silently throwing away a real difference.")
    print()
    drop_dist = []
    columns_after_step_a = precrash.shape[1]
    print(f"RESULT: had {columns_before_this_cell} columns, removed 0, "
          f"still {columns_after_step_a} columns, this was a check only.")

# --- 2b. coordinate pair, already resolved by Part 2's pre-crash filter ---
print()
print("Latitude/longitude vs the weather-lookup coordinates:")
for real_col, process_col in [("latitude", "archive_retrieved_latitude"), ("longitude", "archive_retrieved_longitude")]:
    if real_col in precrash.columns and process_col in full.columns:
        merged_check = pd.concat([full[real_col], full[process_col]], axis=1).dropna()
        max_diff = (merged_check[real_col] - merged_check[process_col]).abs().max()
        print(f"  {real_col} vs {process_col}: max difference {max_diff:.6f} "
              f"({'already excluded from candidates' if process_col not in precrash.columns else 'STILL PRESENT, check Part 2'})")
        provenance_log.append({
            "canonical_column": real_col,
            "original_duplicate_columns": process_col,
            "reason": "The weather-lookup copy is process metadata, already excluded in Part 2, not re-added here",
        })
print()
print(f"RESULT: had {columns_after_step_a} columns, removed 0, "
      f"still {columns_after_step_a} columns, this was a check only.")

# --- 2c. date pair, already resolved by Part 2's pre-crash filter ---
print()
print("accidentdate_parsed vs weather_date:")
if "accidentdate_parsed" in full.columns and "weather_date" in full.columns:
    leftover_dates = [c for c in ["accidentdate_parsed", "weather_date"] if c in precrash.columns]
    if leftover_dates:
        print(f"  Warning, these should have been excluded but were not: {', '.join(leftover_dates)}")
    else:
        print(f"  Both are excluded in Part 2 as timestamps, not predictive inputs.")
        print(f"  Double-checked: neither one snuck back into the kept columns.")
        provenance_log.append({
            "canonical_column": "(neither kept)",
            "original_duplicate_columns": "accidentdate_parsed, weather_date",
            "reason": "Both are timestamps, excluded in Part 2 as non-predictive, not a modeling duplicate to resolve here",
        })
print()
print(f"RESULT: had {columns_after_step_a} columns, removed 0, "
      f"still {columns_after_step_a} columns, this was a check only.")

provenance_df = pd.DataFrame(provenance_log)
print()
print("Summary, what happened to each duplicate:")
for row in provenance_log:
    print(f"  {row['original_duplicate_columns']}")
    print(f"    -> {row['canonical_column']}")
    print(f"    Why: {row['reason']}")
    print()


### Exact duplicate wind-direction columns


In [ ]:
wind_bases = ["winddirection_10m_sin", "winddirection_10m_cos", "winddirection_100m_sin", "winddirection_100m_cos"]
wind_duplicates = []
wind_unclear = []
wind_already_gone = []
n_pairs_checked = 0
print("Confirming each rolling wind field is identical to its unlagged source:")
for base in wind_bases:
    unlagged_col = f"archive.hourly.{base}"
    for window in ["3h", "6h"]:
        rolling_col = f"archive.hourly.{base}_avg_last_{window}"
        if unlagged_col in precrash.columns and rolling_col in precrash.columns:
            n_pairs_checked += 1
            both_present = precrash[[unlagged_col, rolling_col]].dropna()
            if len(both_present) == 0:
                print(f"  {rolling_col} vs {unlagged_col}: no rows with both filled in, cannot confirm, keeping it for now")
                wind_unclear.append(rolling_col)
                continue
            max_diff = (both_present[unlagged_col] - both_present[rolling_col]).abs().max()
            print(f"  {rolling_col} vs {unlagged_col}: max difference {max_diff}, checked on {len(both_present):,} rows")
            # A difference below 1e-9 is ordinary decimal rounding noise, not a real
            # difference, computers cannot store most decimals with perfect precision.
            if max_diff < 1e-9:
                wind_duplicates.append(rolling_col)
            else:
                wind_unclear.append(rolling_col)
        elif rolling_col not in precrash.columns:
            wind_already_gone.append(rolling_col)

print()
print(f"Summary: 8 rolling wind fields exist in the raw data. {n_pairs_checked} were actually checked here.")
print(f"  Confirmed exact duplicates: {len(wind_duplicates)}")
print(f"  Could not confirm, kept as is: {len(wind_unclear)}")
if wind_already_gone:
    print(f"  Already missing before this cell ran: {len(wind_already_gone)}")
    print(f"    This usually means this cell already ran once before in this session,")
    print(f"    and already removed them. If that was not expected, restart the kernel")
    print(f"    and run every cell from the top, in order, then run this one again.")

if wind_unclear:
    print()
    print("Not dropped, could not confirm as a true duplicate, needs a manual look:")
    for col in wind_unclear:
        print(f"  {col}")

precrash = precrash.drop(columns=wind_duplicates)
print()
print(f"Removed {len(wind_duplicates)} confirmed-duplicate rolling wind columns. Columns remaining: {precrash.shape[1]}")
print(f"RESULT: {len(wind_duplicates)} columns removed from the dataset.")


### Part 5, in one place

A short recap, since two separate checks in a row is easy to lose track of.


In [ ]:
starting_count = columns_before_part5
print(f"Started Part 5 with: {starting_count} columns")
print(f"  Removed {len(fully_empty)} columns that were completely empty, no information in them at all")
print(f"  Removed {len(drop_dist)} columns that were exact duplicates of each other, kept one")
print(f"  Removed {len(wind_duplicates)} columns that were exact duplicates of other wind columns")
print()
print(f"Ended Part 5 with: {precrash.shape[1]} columns")


## Part 6: two smaller data quality notes

Neither of these removes anything from the final list, they're both
worth knowing about either way.

### Cyclist eligibility check, `vehicletype`


In [ ]:
if "vehicletype" in full.columns:
    counts = full["vehicletype"].value_counts(dropna=False)
    print("Real values found in vehicletype, across the full dataset:")
    print(counts.to_string())
    n_missing = full["vehicletype"].isna().sum()
    non_missing_values = full["vehicletype"].dropna().unique()
    if len(non_missing_values) == 1:
        print()
        print(f"Every record that has a value at all says {non_missing_values[0]!r}. Nothing else")
        print(f"ever shows up, which is exactly what you'd expect for a study that only looks at")
        print(f"cyclist crashes in the first place.")
        print()
        print(f"About the {n_missing} rows with nothing recorded here: nothing gets deleted because")
        print(f"of this. Those {n_missing} crashes stay in the dataset exactly as they are, every other")
        print(f"column for those rows is untouched. The only thing that happens is this one column,")
        print(f"vehicletype, was already going to be left out of the model entirely, since it only ever")
        print(f"describes the vehicle after a crash has happened, not something you could know beforehand.")
        print(f"So the missing values here have zero effect on anything, the column was never going")
        print(f"to be used either way.")
    else:
        print()
        print(f"WARNING: more than one vehicle type found: {', '.join(str(v) for v in non_missing_values)}. "
              f"This needs review before assuming every record is a genuine cyclist crash.")
    print()
    print("RESULT: no columns removed here, this was a check only.")


### Pavement width discrepancy


In [ ]:
if "pavement_min_width_m" in precrash.columns and "pavement_avg_width_m" in precrash.columns:
    avg_col = precrash["pavement_avg_width_m"]
    min_col = precrash["pavement_min_width_m"]

    print(f"Distinct values in pavement_avg_width_m: {sorted(avg_col.dropna().unique())}")
    is_all_integer = (avg_col.dropna() % 1 == 0).all()
    print(f"All values are whole numbers: {is_all_integer}")
    print()

    flipped = precrash[min_col > avg_col]
    print(f"Rows where min width exceeds average width: {len(flipped):,} of {len(precrash):,} "
          f"({len(flipped)/len(precrash)*100:.1f}%)")

    if len(flipped) > 0:
        excess = (flipped["pavement_min_width_m"] - flipped["pavement_avg_width_m"])
        print()
        print("How much bigger min is than average, in those affected rows:")
        print(f"  Smallest difference: {excess.min():.2f} metres")
        print(f"  Typical difference: {excess.median():.2f} metres")
        print(f"  Average difference: {excess.mean():.2f} metres")
        print(f"  Largest difference: {excess.max():.2f} metres")

        print()
        print("Best-guess interpretation, to confirm with whoever built this field, not a final answer:")
        print("pavement_avg_width_m only ever takes whole numbers 1 to 8, which does not look like a")
        print("real, continuously measured average width in metres, most real streets would not cluster")
        print("into exactly 8 round metre bands. This looks more like a categorised width BAND, for")
        print("example band 1 to band 8 standing in for a range of widths, rather than a true average.")
        print("If that is right, comparing it directly against pavement_min_width_m, which does look like")
        print("a real continuous measurement, is comparing two different kinds of quantity, a band label")
        print("against a real distance, which would fully explain why min can appear to exceed \"average\".")
        print("This needs confirming against the field's actual source definition before either column")
        print("is used or renamed in modelling.")
    print()
    print("RESULT: no columns removed here, this was a check only.")


## Part 7: two risks worth checking directly, not just describing

### Real check 1: does the weather reading ever come from after the crash?


In [ ]:
weather_check = full[["accidentdate_parsed", "weather_date"]].dropna().copy()
weather_check["accidentdate_parsed"] = pd.to_datetime(weather_check["accidentdate_parsed"])
weather_check["weather_date"] = pd.to_datetime(weather_check["weather_date"])

after_crash = weather_check["weather_date"] > weather_check["accidentdate_parsed"]
n_checked = len(weather_check)
n_after = after_crash.sum()

print(f"Rows checked: {n_checked:,}")
print(f"Weather reading timestamped AFTER the crash: {n_after:,} ({n_after/n_checked*100:.2f}%)")
if n_after == 0:
    print("Every single weather reading is at or before its crash. This family is confirmed clean.")
else:
    print("Some weather readings come from after the crash. This is a real problem, worth fixing")
    print("before relying on this family, not something to note and move past.")
print()
print("RESULT: no columns removed here, this was a check only.")


### Real check 2: does the crash-history count ever reach into the future?

**What this column is meant to mean**: for a given crash, "how many
other crashes happened within 100 metres of here, in the year before
this one." That's a genuinely useful, genuinely pre-crash fact, a
location's crash history is knowable in advance. The concern is purely
about the counting itself: did whoever built this column definitely
stop counting the moment they reached this crash's own date, or could
it have accidentally reached one day past that line and counted
something it shouldn't have known about yet.

**How this checks that, in plain steps**:
1. Pick a real crash from the data, with its real location and its real date.
2. Look at every other crash in the whole dataset, and count how many
   sit within 100 metres of this one, and happened strictly before it,
   within the last 365 days.
3. That recount is what the number is SUPPOSED to be, worked out from
   scratch, with no chance of it accidentally including anything from
   the future, since the rule is written to only look backward.
4. Compare that fresh, from-scratch number against whatever number is
   already sitting in the file for that same crash.
5. Repeat this for thousands of real crashes, not just one, so a rare
   problem isn't missed by bad luck.

If the two numbers match every time, the column is doing exactly what
it says. If they don't match, that tells you something is different
between how the real column was built and this simple, safe rule, even
if it doesn't say for certain which one is wrong.


In [ ]:
N_SPOT_CHECK = 5000  # generous on purpose: a rare, occasional leakage bug could easily be
# missed entirely by a small sample, this stays fast, well under a minute, even at full scale
spot_check_sample = full.dropna(subset=["easting", "northing", "accidentdate_parsed", "past_accidents_100m_1y"]).sample(
    min(N_SPOT_CHECK, len(full)), random_state=11
)

all_coords = full[["easting", "northing"]].values
all_dates = pd.to_datetime(full["accidentdate_parsed"]).values.astype("datetime64[D]")

mismatches = []
for idx, row in spot_check_sample.iterrows():
    this_coord = np.array([row["easting"], row["northing"]])
    this_date = np.datetime64(pd.to_datetime(row["accidentdate_parsed"]), "D")
    dist = np.sqrt(((all_coords - this_coord) ** 2).sum(axis=1))
    within_100m = dist <= 100
    within_1y_before = (all_dates < this_date) & (all_dates >= this_date - np.timedelta64(365, "D"))
    recomputed = int((within_100m & within_1y_before).sum())
    stored = row["past_accidents_100m_1y"]
    if recomputed != stored:
        mismatches.append({"collision_id": row.get("collision_id", idx), "stored": stored, "recomputed": recomputed})

print(f"Collisions spot-checked: {len(spot_check_sample)}")
print(f"Mismatches between the stored count and an independent recount: {len(mismatches)}")
if mismatches:
    mismatch_df = pd.DataFrame(mismatches)
    mismatch_df["difference"] = mismatch_df["recomputed"] - mismatch_df["stored"]
    n_higher = (mismatch_df["difference"] > 0).sum()
    n_lower = (mismatch_df["difference"] < 0).sum()
    print()
    print("Which way the mismatches go, this matters a lot for what it actually means:")
    print(f"  Recount found MORE crashes than stored: {n_higher} times")
    print(f"  Recount found FEWER crashes than stored: {n_lower} times")
    print()
    if n_higher > n_lower * 2:
        print("Most mismatches go one way: the stored number is usually too LOW, not too high.")
        print("A leak from the future would normally make the stored number too HIGH instead,")
        print("counting things it should not yet know about. This pattern points more toward")
        print("a difference in exactly where the counting window starts or ends, for example")
        print("whether a crash on the exact boundary day is counted or not, rather than a real")
        print("leak. Still worth confirming against how this column was originally built,")
        print("not something to dismiss just because the direction looks less alarming.")
    print()
    print("Every collision where the two numbers did not match:")
    for _, row in mismatch_df.iterrows():
        direction = "higher" if row["difference"] > 0 else "lower"
        print(f"  Collision {row['collision_id']}: file says {row['stored']}, "
              f"recount says {row['recomputed']} ({abs(row['difference']):.0f} {direction})")
else:
    print("Every recomputed count matches the stored one exactly, in this sample.")
    print("This family checks out, at least for the collisions actually tested here.")
print()
print("RESULT: no columns removed here, this was a check only.")


## Part 8: the final feature list, summarised by family

This is the outcome of everything above, organised the same way as the
rest of this project, one family per row, with the real column count
and the real reason each family is genuinely pre-crash.


In [ ]:
family_rows = []
for col in precrash.columns:
    if col == OUTCOME_COLUMN:
        continue
    family_rows.append({"feature": col, "family": categorize(col)})
family_df = pd.DataFrame(family_rows)

family_summary = family_df.groupby("family").size().rename("column_count").reset_index()
family_summary["justification"] = family_summary["family"].map(FAMILY_JUSTIFICATIONS)
family_summary = family_summary.sort_values("column_count", ascending=False).reset_index(drop=True)

total_final_features = family_summary["column_count"].sum()
print(f"This is the real, final list: {total_final_features} features, after both the initial")
print("filter and all five cleaning steps, grouped into these families:")
print()
for _, row in family_summary.iterrows():
    print(f"{row['family']}  ({row['column_count']} columns)")
    print(f"  {row['justification']}")
    print()


## Part 9: save the cleaned pre-crash dataset


In [ ]:
print(f"Final cleaned pre-crash dataset: {precrash.shape[0]:,} rows, {precrash.shape[1]} columns")
precrash.to_csv(CLEAN_OUTPUT_CSV, index=False)
provenance_df.to_csv(os.path.join(INPUT_FOLDER, "column_provenance_log.csv"), index=False)
print(f"Saved cleaned dataset to: {CLEAN_OUTPUT_CSV}")
print(f"Saved provenance log to: {os.path.join(INPUT_FOLDER, 'column_provenance_log.csv')}")


## Part 10: a granular, per-feature diagnostic, ready for detailed questions

The family view above is what you'd present as a summary. This is the
backup for the question it's likely to provoke, what about feature Z
specifically. Every one of the final kept features gets profiled here.


In [ ]:
def profile_feature(df_, col, family):
    series = df_[col]
    n = len(series)
    n_missing = series.isna().sum()
    profile = {
        "feature": col,
        "family": family,
        "data_type": "Number" if pd.api.types.is_numeric_dtype(series) else "Text/category",
        "missing_count": n_missing,
        "missing_pct": round(n_missing / n * 100, 2),
        "distinct_count": series.nunique(dropna=True),
    }
    non_null = series.dropna()
    if pd.api.types.is_numeric_dtype(series) and len(non_null) > 0:
        profile["min"] = non_null.min()
        profile["max"] = non_null.max()
        profile["mean"] = round(non_null.mean(), 4)
        profile["median"] = non_null.median()
        profile["std_dev"] = round(non_null.std(), 4) if len(non_null) > 1 else 0
        mode_vals = non_null.mode()
        profile["most_common_value"] = mode_vals.iloc[0] if len(mode_vals) > 0 else None
        profile["real_categories_or_examples"] = ", ".join(str(v) for v in non_null.sample(min(5, len(non_null)), random_state=1))
    else:
        profile["min"] = None
        profile["max"] = None
        profile["mean"] = None
        profile["median"] = None
        profile["std_dev"] = None
        mode_vals = non_null.mode()
        profile["most_common_value"] = mode_vals.iloc[0] if len(mode_vals) > 0 else None
        if profile["distinct_count"] <= 15:
            profile["real_categories_or_examples"] = ", ".join(str(v) for v in sorted(non_null.unique(), key=str))
        else:
            profile["real_categories_or_examples"] = ", ".join(str(v) for v in non_null.sample(min(5, len(non_null)), random_state=1)) + ", ... (too many distinct values to list all)"
    return profile

# Every one of the 463 original columns gets profiled here now, not just
# the 373 kept, using the same real family logic as everywhere else:
# the exclusion category for anything excluded, categorize() otherwise.
col_to_excl_category_diag = {}
for category, cols in EXCLUDED_COLUMNS.items():
    for c in cols:
        col_to_excl_category_diag[c] = category

all_feature_rows = []
for col in full.columns:
    if col == OUTCOME_COLUMN:
        continue
    if col in col_to_excl_category_diag:
        family = col_to_excl_category_diag[col]
    elif col in CORRELATION_EXCLUDED:
        family = categorize(col)
    else:
        family = categorize(col)
    all_feature_rows.append(profile_feature(full, col, family))

feature_diagnostics = pd.DataFrame(all_feature_rows)
feature_diagnostics["status"] = feature_diagnostics["feature"].apply(
    lambda c: "Kept" if c in precrash.columns else "Excluded"
)
# severity_corr comes from Part 3, computed there for every numeric column
# not excluded by name. Blank for text columns, and for anything excluded
# by name in Part 2, since correlating a post-crash column with the
# outcome is not meaningful, it is excluded for timing, not relevance.
# Still shown for the 25 correlation-redundant columns and the exact
# duplicates removed in Part 5, since seeing their correlation is part of
# understanding why they duplicate something already kept.
feature_diagnostics["correlation_with_severity"] = feature_diagnostics["feature"].map(severity_corr)

print(f"Profiled all {len(feature_diagnostics)} original features, kept and excluded alike.")
n_kept_diag = (feature_diagnostics["status"] == "Kept").sum()
n_excluded_diag = (feature_diagnostics["status"] == "Excluded").sum()
n_numbers = (feature_diagnostics["data_type"] == "Number").sum()
n_text = (feature_diagnostics["data_type"] == "Text/category").sum()
n_any_missing = (feature_diagnostics["missing_count"] > 0).sum()
print(f"  Kept: {n_kept_diag}")
print(f"  Excluded: {n_excluded_diag}")
print(f"  Numbers: {n_numbers}")
print(f"  Text/category: {n_text}")
print(f"  Features with any missing values: {n_any_missing}")


### The features actually worth a second look

Rather than leave you to scroll through hundreds of rows looking for problems,
here are the ones most likely to need attention: anything that never
actually changes, which means it can't teach a model anything, and the
handful of features missing the most data.


In [ ]:
kept_diagnostics = feature_diagnostics[feature_diagnostics["status"] == "Kept"]
no_variation = kept_diagnostics[kept_diagnostics["distinct_count"] <= 1]
print(f"Features that only ever take one single value: {len(no_variation)}")
if len(no_variation) > 0:
    print("These carry no information at all for a model to learn from, since every crash")
    print("looks identical on this one measurement. Worth dropping, not just noting.")
    for _, row in no_variation.iterrows():
        print(f"  {row['feature']} ({row['family']}), always: {row['most_common_value']}")
print()

most_missing = kept_diagnostics.sort_values("missing_pct", ascending=False).head(10)
print("The 10 features with the most missing data:")
for _, row in most_missing.iterrows():
    print(f"  {row['feature']} ({row['family']}): {row['missing_pct']}% missing")


### Two quick pictures of the same list

Two simple charts: how many features sit in each family, and which
families have the most missing data, since that second question is
easy to answer with a number but much easier to actually see as a
picture.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Chart 1: how many features per family
family_counts_chart = feature_diagnostics[feature_diagnostics["status"] == "Kept"]["family"].value_counts().sort_values(ascending=True)
ax = axes[0]
bars = ax.barh(family_counts_chart.index, family_counts_chart.values, color="#4C72B0")
for b, v in zip(bars, family_counts_chart.values):
    ax.text(v + 1, b.get_y() + b.get_height() / 2, str(v), va="center", fontsize=9)
ax.set_title("How many features in each family", fontsize=12, fontweight="bold")
ax.set_xlabel("Number of features")
ax.spines[["top", "right"]].set_visible(False)

# Chart 2: average missing percentage per family, a different, useful question
missing_by_family = feature_diagnostics[feature_diagnostics["status"] == "Kept"].groupby("family")["missing_pct"].mean().sort_values(ascending=True)
ax = axes[1]
bars = ax.barh(missing_by_family.index, missing_by_family.values, color="#C44E52")
for b, v in zip(bars, missing_by_family.values):
    ax.text(v + 0.5, b.get_y() + b.get_height() / 2, f"{v:.1f}%", va="center", fontsize=9)
ax.set_title("Average share of missing data, by family", fontsize=12, fontweight="bold")
ax.set_xlabel("Average % missing")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()


### Exporting the full diagnostic to Excel


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

wb_diag = Workbook()
ws_diag = wb_diag.active
ws_diag.title = "Feature diagnostics"

export_cols = ["feature", "status", "family", "correlation_with_severity", "data_type",
               "missing_count", "missing_pct", "distinct_count", "min", "max", "mean",
               "median", "std_dev", "most_common_value", "real_categories_or_examples"]
export_headers = ["Feature", "Status", "Family", "Correlation with severity", "Type",
                   "Missing (count)", "Missing (%)", "Distinct values", "Min", "Max", "Mean",
                   "Median", "Spread (std dev)", "Most common value", "Real categories or example values"]

header_font = Font(name="Arial", bold=True, color="FFFFFF")
header_fill = PatternFill(start_color="4C72B0", end_color="4C72B0", fill_type="solid")
body_font = Font(name="Arial")
high_missing_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
high_correlation_fill = PatternFill(start_color="D9C6EC", end_color="D9C6EC", fill_type="solid")

# A short legend, right at the top of the sheet, so both highlights
# explain themselves the moment the file is opened, not just in a print
# statement that might not be seen.
ws_diag["A1"] = ("Every one of the 463 original columns, kept and excluded alike. "
                 "Pink rows have more than 20% missing data. Purple rows correlate with the "
                 f"real outcome more strongly than {OUTCOME_OUTLIER_THRESHOLD:.2f}, unusually "
                 "strong given how weak every genuine relationship in this project has been so far.")
ws_diag["A1"].font = Font(name="Arial", bold=True, italic=True, color="9C0006")
ws_diag.merge_cells(f"A1:{get_column_letter(len(export_cols))}1")

header_row = 3
for col_idx, h in enumerate(export_headers, start=1):
    cell = ws_diag.cell(row=header_row, column=col_idx, value=h)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", wrap_text=True)

export_df_diag = feature_diagnostics.sort_values(["status", "family", "feature"], ascending=[False, True, True])[export_cols].reset_index(drop=True)
first_data_row = header_row + 1
for i, row in enumerate(export_df_diag.itertuples(index=False)):
    row_idx = first_data_row + i
    for col_idx, value in enumerate(row, start=1):
        cell = ws_diag.cell(row=row_idx, column=col_idx)
        cell.value = round(value, 4) if isinstance(value, float) else value
        cell.font = body_font
    missing_pct_val = row[export_cols.index("missing_pct")]
    corr_val = row[export_cols.index("correlation_with_severity")]
    if pd.notna(corr_val) and abs(corr_val) > OUTCOME_OUTLIER_THRESHOLD:
        for col_idx in range(1, len(export_cols) + 1):
            ws_diag.cell(row=row_idx, column=col_idx).fill = high_correlation_fill
    elif missing_pct_val is not None and missing_pct_val > 20:
        for col_idx in range(1, len(export_cols) + 1):
            ws_diag.cell(row=row_idx, column=col_idx).fill = high_missing_fill

for col_idx, col_name in zip(range(1, len(export_cols) + 1), export_cols):
    if col_name == "real_categories_or_examples":
        width = 60
    else:
        sample_vals = export_df_diag[col_name].map(lambda x: str(x) if pd.notna(x) else "")
        width = min(max(sample_vals.map(len).max(), len(col_name)) + 3, 35)
    ws_diag.column_dimensions[get_column_letter(col_idx)].width = width

ws_diag.freeze_panes = f"A{first_data_row}"
ws_diag.auto_filter.ref = f"A{header_row}:{get_column_letter(len(export_cols))}{first_data_row + len(export_df_diag) - 1}"

# ==================================================================
# SHEET 2: the full journey, every one of the 463 original columns,
# through every real cleaning check, one row per column. Reuses
# review_df from Part 3 directly, so the reasoning shown here is
# exactly what was already checked, not a second, separate telling.
# ==================================================================
ws_journey = wb_diag.create_sheet("Full column journey")

ws_journey["A1"] = ("Every original column: its family, whether it was kept, and the full reason why, "
                     "including anything flagged for the team to review even though it was kept.")
ws_journey["A1"].font = Font(name="Arial", bold=True, italic=True)
ws_journey.merge_cells("A1:G1")

col_to_excl_category = {}
for category, cols in EXCLUDED_COLUMNS.items():
    for c in cols:
        col_to_excl_category[c] = category

journey_headers = ["Column", "Family", "Correlation with severity", "Status", "Why (category)", "Why (detail)", "Flagged for the team to review?"]
header_row_j = 3
for col_idx, h in enumerate(journey_headers, start=1):
    cell = ws_journey.cell(row=header_row_j, column=col_idx, value=h)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal="center", wrap_text=True)

# review_df comes from Part 3, one row per column with its suggested_action
# already worked out there, this reuses that directly rather than
# re-deriving the same reasoning a second time in a different form.
review_lookup = review_df.set_index("feature")["suggested_action"].to_dict() if "review_df" in dir() else {}

journey_rows = []
for col in full.columns:
    if col == OUTCOME_COLUMN:
        journey_rows.append([col, "(outcome)", None, "Outcome", "Outcome", "This is the outcome being predicted, not an input", "No"])
        continue

    is_postcrash = col in col_to_excl_category
    is_empty = col in fully_empty
    is_redundant_step5 = col in drop_dist or col in wind_duplicates
    is_redundant_step4 = col in CORRELATION_EXCLUDED
    kept = col in precrash.columns
    family = col_to_excl_category[col] if is_postcrash else categorize(col)

    # A short, filterable tag, so the team can sort or filter by the KIND
    # of reason in Excel, rather than reading the full sentence every time.
    if is_postcrash:
        why_category = "Post-crash info"
        why = f"Excluded, post-crash info, not knowable before a crash ({col_to_excl_category[col]})"
    elif is_empty:
        why_category = "Empty"
        why = "Excluded, 100% empty, no real information in this column at all"
    elif is_redundant_step4:
        why_category = "Correlation"
        why = f"Excluded, confirmed redundant by direct correlation: {CORRELATION_EXCLUDED[col]}"
    elif is_redundant_step5:
        why_category = "Duplicate"
        why = "Excluded, an exact duplicate of another kept column, confirmed by direct comparison"
    elif kept:
        why_category = "Kept"
        why = f"Kept, genuinely pre-crash, part of the {family} family"
    else:
        why_category = "Other"
        why = "Excluded for another reason, check manually"

    flagged_note = "No"
    if kept and col in review_lookup and review_lookup[col] != "KEEP":
        flagged_note = review_lookup[col]
        # A kept row can still carry its own flag reason, shown as a second
        # tag rather than overwriting "Kept", since both things are true.
        if "identifier" in flagged_note:
            why_category = "Kept, flagged: identifier"
        elif "duplicate" in flagged_note:
            why_category = "Kept, flagged: correlation"
        elif "never varies" in flagged_note:
            why_category = "Kept, flagged: no variation"
        elif "real outcome" in flagged_note:
            why_category = "Kept, flagged: outcome link"
        else:
            why_category = "Kept, flagged"

    journey_rows.append([
        col, family, severity_corr.get(col), "Kept" if kept else "Excluded", why_category, why, flagged_note,
    ])

first_data_row_j = header_row_j + 1
kept_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
excluded_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
flagged_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
for i, row_vals in enumerate(journey_rows):
    r = first_data_row_j + i
    for col_idx, val in enumerate(row_vals, start=1):
        cell = ws_journey.cell(row=r, column=col_idx)
        cell.value = round(val, 4) if isinstance(val, float) else val
        cell.font = body_font
    if row_vals[3] == "Kept" and row_vals[6] != "No":
        fill = flagged_fill
    elif row_vals[3] == "Kept":
        fill = kept_fill
    elif row_vals[3] == "Outcome":
        fill = None
    else:
        fill = excluded_fill
    if fill is not None:
        for col_idx in range(1, len(journey_headers) + 1):
            ws_journey.cell(row=r, column=col_idx).fill = fill

for col_idx, width in zip(range(1, len(journey_headers) + 1), [40, 32, 20, 12, 24, 65, 50]):
    ws_journey.column_dimensions[get_column_letter(col_idx)].width = width

ws_journey.freeze_panes = f"A{first_data_row_j}"
ws_journey.auto_filter.ref = f"A{header_row_j}:{get_column_letter(len(journey_headers))}{first_data_row_j + len(journey_rows) - 1}"

n_kept_journey = sum(1 for r in journey_rows if r[3] == "Kept")
n_flagged_journey = sum(1 for r in journey_rows if r[3] == "Kept" and r[6] != "No")
print(f"Full column journey: {len(journey_rows)} original columns, {n_kept_journey} kept in the final model, "
      f"{n_flagged_journey} of those kept but flagged for a second look.")
print("Green rows were kept and clean, yellow rows were kept but flagged, pink rows were excluded, "
      "with the real reason and family in the columns before it.")

diag_out_path = os.path.join(INPUT_FOLDER, "feature_diagnostics.xlsx")
wb_diag.save(diag_out_path)
print(f"Saved {len(export_df_diag)} kept features, and the full {len(journey_rows)}-column journey, to: {diag_out_path}")
print("Rows with more than 20% missing, in the first sheet, are highlighted pink, worth a second look.")
print("That explanation is also written directly at the top of the sheet itself.")


### A quick lookup for a single feature, right here in the notebook

Run this with any feature name to get the same answer live, without
switching to the Excel file.


In [ ]:
def explain_feature(col):
    kept_only = feature_diagnostics[feature_diagnostics["status"] == "Kept"]
    if col not in kept_only["feature"].values:
        print(f"{col!r} is not in the kept pre-crash features, either it was excluded, or the name is not quite right.")
        return
    row = kept_only[kept_only["feature"] == col].iloc[0]
    feature_name = row["feature"]
    family_name = row["family"]
    data_type = row["data_type"]
    missing_count = row["missing_count"]
    missing_pct = row["missing_pct"]
    distinct_count = row["distinct_count"]
    print(f"Feature: {feature_name}")
    print(f"Family: {family_name}")
    print(f"Type: {data_type}")
    print(f"Missing: {missing_count:,} ({missing_pct}%)")
    print(f"Distinct values: {distinct_count:,}")
    if data_type == "Number":
        min_val, max_val, mean_val, median_val, std_val = row["min"], row["max"], row["mean"], row["median"], row["std_dev"]
        print(f"Range: {min_val} to {max_val}, mean {mean_val}, median {median_val}, spread (std dev) {std_val}")
    most_common = row["most_common_value"]
    print(f"Most common value: {most_common}")
    examples = row["real_categories_or_examples"]
    print(f"Real values: {examples}")

# Example, replace with whatever feature name comes up:
explain_feature("speedlimit_clean")


## Part 11: run the Bikeable model across all available images

Same proven scoring pipeline already used and validated elsewhere in
this project, run here fresh so the resulting scores can be merged
into this cleaned dataset directly.


In [ ]:
SEG_OBJ_PATH = os.path.join(INPUT_FOLDER, "processed_features_conf_0.5_filled_metadata_no_collision.csv")
STATS_PATH = os.path.join(INPUT_FOLDER, "image_stats_processed_final_grids_no_collision.csv")
STREETVIEW_META_PATH = os.path.join(INPUT_FOLDER, "streetview_metadata_with_collision.csv")
MODEL_PATH = os.path.join(INPUT_FOLDER, "lightgbm_gpt_final.pkl")

import joblib

loaded = joblib.load(MODEL_PATH)
trained_model = loaded["model"]
trained_feats = loaded["features"]
BIN_LABELS = ["Low", "Medium", "High"]
print(f"Model expects {len(trained_feats)} features.")

seg_obj_header = pd.read_csv(SEG_OBJ_PATH, nrows=0).columns.tolist()
stats_header = pd.read_csv(STATS_PATH, nrows=0).columns.tolist()

composite_source_cols = [
    "seg_Bike Lane", "seg_Bike Rack", "seg_Bicycle", "seg_Bicyclist", "seg_Other Rider", "seg_Wheeled Slow",
    "seg_Car", "seg_Truck", "seg_Bus", "seg_Motorcycle", "seg_Other Vehicle",
    "seg_Barrier", "seg_Guard Rail", "seg_Fence", "seg_Wall", "seg_Tunnel", "seg_Bridge",
    "seg_Vegetation", "seg_Terrain", "seg_Mountain", "seg_Water",
    "seg_Billboard", "seg_Street Light", "seg_Traffic Sign (Front)", "seg_CCTV Camera", "seg_Building",
]
direct_feats_seg = [f for f in trained_feats if f in seg_obj_header]
area_cols_seg = [c for c in seg_obj_header if c.startswith("obj_") and "_area_" in c]
keys = ["image_name", "pano_id"]
needed_seg_cols = sorted(set(keys + composite_source_cols + direct_feats_seg + area_cols_seg))
needed_seg_cols = [c for c in needed_seg_cols if c in seg_obj_header]

raw = pd.read_csv(SEG_OBJ_PATH, usecols=needed_seg_cols,
                   dtype={c: "float32" for c in needed_seg_cols if c not in keys})

direct_feats_stats = [f for f in trained_feats if f in stats_header]
needed_stats_cols = sorted(set(keys + direct_feats_stats))
needed_stats_cols = [c for c in needed_stats_cols if c in stats_header]
img_stats = pd.read_csv(STATS_PATH, usecols=needed_stats_cols,
                         dtype={c: "float32" for c in needed_stats_cols if c not in keys})

overlap_cols = [c for c in img_stats.columns if c not in keys and c in raw.columns]
if overlap_cols:
    img_stats = img_stats.drop(columns=overlap_cols)

photo_scores = raw.merge(img_stats, on=keys, how="inner", validate="one_to_one")
del raw, img_stats
print(f"Merged photo-level rows: {len(photo_scores):,}")

def safe_sum(df_, cols):
    present = [c for c in cols if c in df_.columns]
    return df_[present].fillna(0).sum(axis=1) if present else pd.Series(0, index=df_.index)

photo_scores["comp_cycling_infra"]      = safe_sum(photo_scores, ["seg_Bike Lane", "seg_Bike Rack", "seg_Bicycle", "seg_Bicyclist", "seg_Other Rider", "seg_Wheeled Slow"])
photo_scores["comp_motor_load"]         = safe_sum(photo_scores, ["seg_Car", "seg_Truck", "seg_Bus", "seg_Motorcycle", "seg_Other Vehicle"])
photo_scores["comp_separation"]         = safe_sum(photo_scores, ["seg_Barrier", "seg_Guard Rail", "seg_Fence", "seg_Wall", "seg_Tunnel", "seg_Bridge"])
photo_scores["comp_natural_openness"]   = safe_sum(photo_scores, ["seg_Vegetation", "seg_Terrain", "seg_Mountain", "seg_Water"])
photo_scores["comp_commercial_density"] = safe_sum(photo_scores, ["seg_Billboard", "seg_Street Light", "seg_Traffic Sign (Front)", "seg_CCTV Camera"])
photo_scores["comp_built_density"]      = safe_sum(photo_scores, ["seg_Building", "seg_Wall", "seg_Bridge"])
for suffix in ["cycling_infra", "motor_load", "separation", "natural_openness", "commercial_density", "built_density"]:
    photo_scores[f"comp_has_{suffix}"] = (photo_scores[f"comp_{suffix}"] > 0).astype(int)
photo_scores["comp_nature_vs_built"]  = photo_scores["comp_natural_openness"] / (photo_scores["comp_built_density"] + 0.001)
photo_scores["comp_terrain_vs_motor"] = safe_sum(photo_scores, ["seg_Terrain", "seg_Vegetation"]) / (photo_scores["comp_motor_load"] + 0.001)

IMAGE_PIXELS = 640 * 640
area_cols = [c for c in photo_scores.columns if c.startswith("obj_") and "_area_" in c]
if area_cols and photo_scores[area_cols].max().max() > 1.0:
    photo_scores[area_cols] = photo_scores[area_cols] / IMAGE_PIXELS

missing = [f for f in trained_feats if f not in photo_scores.columns]
assert not missing, f"Missing features, stop and check: {missing}"

X_infer = photo_scores[trained_feats].copy()
proba = trained_model.predict_proba(X_infer)
photo_scores = pd.concat([photo_scores, pd.DataFrame({
    "pred_class": np.argmax(proba, axis=1),
    "proba_low": proba[:, 0],
    "proba_medium": proba[:, 1],
    "proba_high": proba[:, 2],
}, index=photo_scores.index)], axis=1)
photo_scores["pred_label"] = photo_scores["pred_class"].map({0: "Low", 1: "Medium", 2: "High"})

photo_scores["point_id"] = photo_scores["image_name"].str.extract(r"^(\d+)_")[0].astype(int)
sv_meta = pd.read_csv(STREETVIEW_META_PATH, usecols=["image_name", "collision_id"])
sv_meta = sv_meta.rename(columns={"image_name": "point_id"})
sv_meta["point_id"] = sv_meta["point_id"].astype(int)
photo_scores = photo_scores.merge(sv_meta, on="point_id", how="left")

print(f"All {len(photo_scores):,} photos scored and linked to their collision.")


## Part 11b: checking which direction the classes actually run

This checks the class against two things already computed for every
photo, how much cycling infrastructure is visible, and how much motor
traffic is visible. A genuine risk measure should go up with traffic
and down with cycling infrastructure. A genuine safety measure would
run the other way. This settles which one is actually true here,
using real evidence rather than the label alone.


In [ ]:
direction_check = photo_scores[["pred_class", "proba_high", "comp_cycling_infra", "comp_motor_load"]].copy()

corr_infra = direction_check["pred_class"].corr(direction_check["comp_cycling_infra"])
corr_motor = direction_check["pred_class"].corr(direction_check["comp_motor_load"])

print(f"Correlation between the class and visible cycling infrastructure: {corr_infra:+.4f}")
print(f"Correlation between the class and visible motor traffic: {corr_motor:+.4f}")
print()

if corr_motor > 0 and corr_infra < 0:
    print("The High class goes up with traffic and down with cycling infrastructure.")
    print("That is what a risk measure looks like. In this project, High means")
    print("high risk, dangerous looking, not Paper 1's original high safety meaning.")
    HIGH_MEANS_DANGEROUS = True
elif corr_motor < 0 and corr_infra > 0:
    print("The High class goes up with cycling infrastructure and down with traffic.")
    print("That matches Paper 1's original meaning directly. High really does mean")
    print("high safety here, safe looking. Low means low safety, dangerous looking.")
    HIGH_MEANS_DANGEROUS = False
else:
    print("The two checks do not point the same way clearly, this needs a manual look")
    print("before trusting either direction.")
    HIGH_MEANS_DANGEROUS = None

print()
print(f"HIGH_MEANS_DANGEROUS = {HIGH_MEANS_DANGEROUS}")

if HIGH_MEANS_DANGEROUS is False:
    print()
    print("Important consequence: \"worst case\", the least safe view, must be built")
    print("from whichever photo has the LOWEST class number, not the highest. Picking")
    print("the highest class number, which earlier work in this project did, actually")
    print("picks the single SAFEST looking photo, the opposite of what \"worst case\"")
    print("is supposed to mean. This gets fixed properly further down.")
elif HIGH_MEANS_DANGEROUS is True:
    print()
    print("Under this direction, picking the highest class number for \"worst case\"")
    print("is already correct, no change needed to that calculation.")


## Part 12: visual review, low, medium, high, and surprising cases

A "surprising" case here means a photo the model rated confidently one
way, whose real, actual collision severity turned out to disagree
strongly, for example rated Low risk but the real crash was Serious or
Fatal, or rated High risk but the real crash was only Slight. These are
exactly the cases worth eyeballing directly, since they are where the
model's judgement and reality diverge the most.


In [ ]:
severity_by_collision = full[["collision_id", "casualty_severity"]].drop_duplicates(subset="collision_id")
review_set = photo_scores.merge(severity_by_collision, on="collision_id", how="left")

print("Picking real examples to show, in two steps:")
print("  1. Low, Medium, and High: a random sample of each, just to see what typical")
print("     photos in each category actually look like.")
print("  2. Surprising cases: photos where the model was very confident one way,")
print("     but the real crash there turned out to disagree strongly.")
print()

# These are candidate POOLS, not the final picks. A lot of specific photos
# will not actually exist in Azure, so a wider pool gets checked for real
# further down, and only ones confirmed to exist get shown, rather than
# picking a handful up front and hoping they happen to be there.
POOL_SIZE = 40
low_pool = review_set[review_set["pred_label"] == "Low"].sample(
    min(POOL_SIZE, (review_set["pred_label"] == "Low").sum()), random_state=1)
medium_pool = review_set[review_set["pred_label"] == "Medium"].sample(
    min(POOL_SIZE, (review_set["pred_label"] == "Medium").sum()), random_state=1)
high_pool = review_set[review_set["pred_label"] == "High"].sample(
    min(POOL_SIZE, (review_set["pred_label"] == "High").sum()), random_state=1)

# A "surprising" case is not just any disagreement. A fixed confidence cutoff
# like 90% does not work reliably here, since one class can end up covering
# the vast majority of all photos, in which case "confidently rated that
# class" stops meaning anything rare. Using the top 1% most confident within
# each direction instead keeps this genuinely rare no matter how lopsided
# the overall class balance turns out to be.
TOP_PERCENT = 1

if HIGH_MEANS_DANGEROUS:
    danger_label, danger_col = "High", "proba_high"
    safe_label, safe_col = "Low", "proba_low"
else:
    danger_label, danger_col = "Low", "proba_low"
    safe_label, safe_col = "High", "proba_high"

safe_cutoff = review_set[safe_col].quantile(1 - TOP_PERCENT / 100)
danger_cutoff = review_set[danger_col].quantile(1 - TOP_PERCENT / 100)

surprising_looked_safe_but_fatal_pool = review_set[
    (review_set[safe_col] >= safe_cutoff) & (review_set["casualty_severity"] == "Fatal")
].sort_values(safe_col, ascending=False)

surprising_looked_dangerous_but_slight_pool = review_set[
    (review_set[danger_col] >= danger_cutoff) & (review_set["casualty_severity"] == "Slight")
].sort_values(danger_col, ascending=False)

print(f"\"Confident\" here means in the most confident 1% of all photos, in that direction.")
print(f"  For looking {safe_label} (safe): at least {safe_cutoff:.0%} sure.")
print(f"  For looking {danger_label} (dangerous): at least {danger_cutoff:.0%} sure.")
print()
print(f"Looked {safe_label} (safe) that confident, but the real crash was Fatal: "
      f"{len(surprising_looked_safe_but_fatal_pool):,} photos in the full dataset")
print(f"Looked {danger_label} (dangerous) that confident, but the real crash was only Slight: "
      f"{len(surprising_looked_dangerous_but_slight_pool):,} photos in the full dataset")
print()

if len(surprising_looked_safe_but_fatal_pool) <= 10:
    print(f"There are only a few of these, so here they are listed directly:")
    for _, row in surprising_looked_safe_but_fatal_pool.iterrows():
        print(f"  Collision {row['collision_id']}, photo {row['image_name']}: "
              f"{row[safe_col]:.0%} confident it looked {safe_label.lower()}")
    print()

# Keep the same variable names used further down, so nothing else in this
# section needs to change, only which real pool gets assigned to each.
surprising_low_but_fatal_pool = surprising_looked_safe_but_fatal_pool
surprising_high_but_slight_pool = surprising_looked_dangerous_but_slight_pool


### Showing the actual photos

Needs the real Azure connection string, the same one used elsewhere in
this project, pasted below in place of the placeholder. Never save or
share this notebook with that value filled in.


In [ ]:
AZURE_CONNECTION_STRING = (
    '...'
    '...'
    '...'
)
BLOB_CONTAINER = "london"
IMAGE_PREFIX = "Accidents/Images/"  # confirmed the real, full photo folder: 239,324 files, exactly matching the scored dataset

if AZURE_CONNECTION_STRING is None:
    AZURE_CONNECTION_STRING = os.environ.get("AZURE_STORAGE_CONNECTION_STRING")
if AZURE_CONNECTION_STRING:
    AZURE_CONNECTION_STRING = AZURE_CONNECTION_STRING.strip()

connection_ok = False
if not AZURE_CONNECTION_STRING or "PASTE_YOUR_ACCOUNT_KEY_HERE" in AZURE_CONNECTION_STRING:
    print("No real Azure connection string set yet, so the actual images cannot be shown here.")
    print("Paste your real key in place of PASTE_YOUR_ACCOUNT_KEY_HERE above, then rerun this cell.")
else:
    import io
    from PIL import Image
    from azure.storage.blob import BlobServiceClient

    blob_svc = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
    container_client = blob_svc.get_container_client(BLOB_CONTAINER)

    def fetch_image(image_name):
        blob_data = container_client.download_blob(IMAGE_PREFIX + image_name, timeout=30).readall()
        return Image.open(io.BytesIO(blob_data)).convert("RGB")

    # A quick, one-time sanity check that the connection and the path are
    # both genuinely working, before spending time fetching real photos.
    connection_ok = False
    try:
        container_client.get_container_properties()
        connection_ok = True
        print(f"Connected to Azure successfully, using the confirmed real photo folder.")
    except Exception as e:
        print(f"Could not connect to Azure: {type(e).__name__}: {e}")
        print("Check the key above is current, then rerun this cell.")

    if connection_ok:
        def show_review_row(ax, row, tag):
            try:
                ax.imshow(fetch_image(row["image_name"]))
            except Exception as e:
                ax.text(0.5, 0.5, f"Could not fetch:\n{type(e).__name__}", ha="center", va="center", transform=ax.transAxes, fontsize=8)
            ax.set_title(f"{tag}\nCollision {row['collision_id']}\nPredicted: {row['pred_label']}  |  Real: {row['casualty_severity']}", fontsize=9)
            ax.axis("off")

        def pick_existing(pool, n_wanted, max_checks=40):
            """Checks candidates from a pool one at a time, in order, and keeps
            the first n_wanted that are actually confirmed to exist in storage,
            rather than picking blindly and hoping. Stops after max_checks either
            way, so this never runs away checking forever."""
            picked_rows = []
            checked = 0
            for _, row in pool.iterrows():
                if len(picked_rows) >= n_wanted or checked >= max_checks:
                    break
                checked += 1
                exists = container_client.get_blob_client(IMAGE_PREFIX + row["image_name"]).exists()
                if exists:
                    picked_rows.append(row)
            return pd.DataFrame(picked_rows), checked

        N_EXAMPLES_WANTED = 5
        print()
        print(f"Picking {N_EXAMPLES_WANTED} real photos for each group, checking each one really exists first:")

        low_examples, n_checked_low = pick_existing(low_pool, N_EXAMPLES_WANTED)
        medium_examples, n_checked_medium = pick_existing(medium_pool, N_EXAMPLES_WANTED)
        high_examples, n_checked_high = pick_existing(high_pool, N_EXAMPLES_WANTED)
        surprising_low_but_fatal, n_checked_slf = pick_existing(surprising_low_but_fatal_pool, N_EXAMPLES_WANTED)
        surprising_high_but_slight, n_checked_shs = pick_existing(surprising_high_but_slight_pool, N_EXAMPLES_WANTED)
        surprising_examples = pd.concat([surprising_low_but_fatal, surprising_high_but_slight])

        results = [
            ("Low", low_examples, n_checked_low),
            ("Medium", medium_examples, n_checked_medium),
            ("High", high_examples, n_checked_high),
            (f"Surprising, looked {safe_label} but Fatal", surprising_low_but_fatal, n_checked_slf),
            (f"Surprising, looked {danger_label} but Slight", surprising_high_but_slight, n_checked_shs),
        ]
        for label, examples, n_checked in results:
            print(f"  {label}: found {len(examples)} of {N_EXAMPLES_WANTED} wanted, checked {n_checked} photos to get there")
        print()

        # Two rows for the surprising cases, one per direction, each labelled
        # clearly, rather than merging them into one oversized row that leaves
        # the other rows padded out with wasted blank space.
        review_groups = [
            ("Low", low_examples), ("Medium", medium_examples), ("High", high_examples),
            (f"SURPRISING: looked {safe_label}, safe, but Fatal", surprising_low_but_fatal),
            (f"SURPRISING: looked {danger_label}, dangerous, but Slight", surprising_high_but_slight),
        ]
        n_rows = len(review_groups)
        n_cols = max(len(g) for _, g in review_groups)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 4.6 * n_rows))

        for row_idx, (tag, group) in enumerate(review_groups):
            for col_idx in range(n_cols):
                ax = axes[row_idx, col_idx] if n_rows > 1 else axes[col_idx]
                if col_idx < len(group):
                    show_review_row(ax, group.iloc[col_idx], tag)
                else:
                    ax.axis("off")

        # Extra top margin and space between rows, since each title here is
        # three lines tall, a plain tight_layout does not leave enough room
        # and the text ends up overlapping the images above it.
        plt.suptitle("Visual review: low, medium, high, and surprising cases", fontsize=13, fontweight="bold", y=1.0)
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.subplots_adjust(hspace=0.6)
        plt.show()


### A closer look at the rare, confidently-safe-but-Fatal photos


In [ ]:
if not connection_ok:
    print("Azure is not connected, run the cell above first with your real key.")
elif len(surprising_low_but_fatal_pool) == 0:
    print("None of these exist in the full dataset right now, nothing to show.")
else:
    n_photos = len(surprising_low_but_fatal_pool)
    print(f"Showing all {n_photos} of the rare, confidently-safe-but-Fatal photos directly.")
    print()

    fig, axes = plt.subplots(1, n_photos, figsize=(4 * n_photos, 5.2))
    if n_photos == 1:
        axes = [axes]

    for col_idx, (_, row) in enumerate(surprising_low_but_fatal_pool.iterrows()):
        ax = axes[col_idx]
        try:
            ax.imshow(fetch_image(row["image_name"]))
        except Exception as e:
            ax.text(0.5, 0.5, f"Could not fetch:\n{type(e).__name__}", ha="center", va="center", transform=ax.transAxes, fontsize=8)
        ax.set_title(f"Collision {row['collision_id']}\n{row['image_name']}\n"
                     f"{row[safe_col]:.0%} confident it looked {safe_label.lower()}", fontsize=9)
        ax.axis("off")

    # Extra top margin, since the three-line title otherwise overlaps the
    # main title above it.
    plt.suptitle("Every rare case: confidently safe looking, but the real crash was Fatal", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout(rect=[0, 0, 1, 0.90])
    plt.show()


## Part 13: building the scenario scores, from one single source

Both scores get built here from the exact same photos scored earlier
in this notebook, so nothing about how the photos were scored can be
the reason for any difference found later.

- **Average**: each photo's 3 probabilities become one weighted value,
  Low = 0, Medium = 1, High = 2, averaged across the 4 photos. A real
  number, already confirmed to match your previously stored file.
- **Worst case**: whichever single photo among the 4 is the least
  safe one. Which end of the scale that actually is depends on the
  direction confirmed in Part 11b, so this is worked out from that
  directly, rather than assumed.

Scenario A and Scenario C, two other ways of turning the 4 photos into
one class, also get built here, since they were part of your original
work too.


In [ ]:
# The average, a real number per collision.
weighted_per_photo = photo_scores["proba_low"] * 0 + photo_scores["proba_medium"] * 1 + photo_scores["proba_high"] * 2
photo_scores["weighted_value"] = weighted_per_photo

# The worst case, also a real number per collision, not just the class of
# whichever photo is worst. Confidence that the single least-safe photo
# actually looks dangerous, using whichever direction Part 11b confirmed.
if HIGH_MEANS_DANGEROUS:
    worst_case_continuous = photo_scores.groupby("collision_id")["proba_high"].max()
else:
    worst_case_continuous = photo_scores.groupby("collision_id")["proba_low"].max()

continuous_scores = photo_scores.groupby("collision_id").agg(
    perceived_safety_average=("weighted_value", "mean"),
    n_photos_scored=("weighted_value", "size"),
).reset_index()
continuous_scores = continuous_scores.merge(
    worst_case_continuous.rename("perceived_safety_worst_case_score").reset_index(),
    on="collision_id", how="left",
)

print(f"Collisions with at least one scored photo: {len(continuous_scores):,}")
print()
print("The average safety score across all of them:")
print(f"  Smallest: {continuous_scores['perceived_safety_average'].min():.3f}")
print(f"  Typical: {continuous_scores['perceived_safety_average'].median():.3f}")
print(f"  Largest: {continuous_scores['perceived_safety_average'].max():.3f}")
print()
print("The worst case safety score across all of them:")
print(f"  Smallest: {continuous_scores['perceived_safety_worst_case_score'].min():.3f}")
print(f"  Typical: {continuous_scores['perceived_safety_worst_case_score'].median():.3f}")
print(f"  Largest: {continuous_scores['perceived_safety_worst_case_score'].max():.3f}")
print()
print(f"Photos per collision, typically {continuous_scores['n_photos_scored'].median():.0f}, "
      f"ranging from {continuous_scores['n_photos_scored'].min():.0f} to {continuous_scores['n_photos_scored'].max():.0f}")


In [ ]:
# The three original scenarios, each a rounded class, recreated from
# the same photo_scores used above. A duplicated index is a real, silent
# risk with the scenario_b lookup below, so this guards against it.
scenario_df = photo_scores.reset_index(drop=True)

scenario_a = scenario_df.groupby("collision_id")[["proba_low", "proba_medium", "proba_high"]].mean()
scenario_a["scenario_a_class"] = scenario_a[["proba_low", "proba_medium", "proba_high"]].values.argmax(axis=1)

# Which end actually is the "worst", the least safe view, depends on which
# direction Part 11b confirmed. If High means dangerous, the least safe view
# is whichever photo has the HIGHEST class number. If High means safe, the
# least safe view is whichever photo has the LOWEST class number instead,
# since that is the one furthest from safe. Getting this backwards means
# picking out the single safest photo while calling it the worst case,
# which is exactly what happened before this was checked properly.
if HIGH_MEANS_DANGEROUS:
    worst_case_idx = scenario_df.groupby("collision_id")["pred_class"].idxmax()
else:
    worst_case_idx = scenario_df.groupby("collision_id")["pred_class"].idxmin()

scenario_b = scenario_df.loc[worst_case_idx][["collision_id", "pred_class"]]
scenario_b = scenario_b.rename(columns={"pred_class": "scenario_b_class"}).set_index("collision_id")

def majority_vote(group):
    counts = group["pred_class"].value_counts()
    top = counts.iloc[0]
    tied = counts[counts == top].index.tolist()
    if len(tied) == 1:
        return tied[0]
    return group[["proba_low", "proba_medium", "proba_high"]].mean().values.argmax()

# include_groups is left out here on purpose, it only exists on newer
# pandas, and this function never touches the grouping column anyway,
# so leaving it out means this runs the same way on any pandas version.
scenario_c = scenario_df.groupby("collision_id").apply(majority_vote).rename("scenario_c_class").to_frame()

scenario_classes = scenario_a[["scenario_a_class"]].join(scenario_b).join(scenario_c).reset_index()

# scenario_b_class is the confirmed-correct worst case, given a plain
# name here so everything from this point on reads clearly.
scenario_classes["perceived_safety_worst_case"] = scenario_classes["scenario_b_class"]

print(f"Collisions with a scenario class: {len(scenario_classes):,}")
print()
print("How often each class comes up, for each of the three scenarios:")
for col in ["scenario_a_class", "perceived_safety_worst_case", "scenario_c_class"]:
    print(f"  {col}: {scenario_classes[col].value_counts().sort_index().to_dict()}")


### Checking the average still lines up with its own class version

The worst case question is already settled, so this just checks that
Scenario A, the rounded class next to the real average number, still
moves the way it should, a high real number should mean a high class.


In [ ]:
all_scenarios = continuous_scores.merge(scenario_classes, on="collision_id", how="inner")

corr = all_scenarios["perceived_safety_average"].corr(all_scenarios["scenario_a_class"])
print(f"Average (real number) vs Scenario A (class): correlation {corr:.4f}")
print()
print("What the real average number looks like within each class:")
class_stats = all_scenarios.groupby("scenario_a_class")["perceived_safety_average"].agg(["mean", "min", "max"])
class_labels = {0: "Class 0 (Low)", 1: "Class 1 (Medium)", 2: "Class 2 (High)"}
for class_val, row in class_stats.iterrows():
    label = class_labels.get(class_val, f"Class {class_val}")
    print(f"  {label}: typically {row['mean']:.3f}, ranging from {row['min']:.3f} to {row['max']:.3f}")


## Part 14: checking this against your previously stored scores

The average is expected to match your old file exactly, as before.
The worst case is a different story now. If Part 11b found that High
actually means safe, the worst case built here correctly points at
the least safe photo, while the old stored file was built by picking
the highest class number regardless, which turns out to mean the
safest photo instead. A mismatch here, in that specific situation, is
the expected, correct outcome, not a problem to chase down, it means
this has been fixed rather than broken.


In [ ]:
OLD_AVERAGE_CSV = os.path.join(INPUT_FOLDER, "collisions_with_perception_average.csv")
OLD_WORSTCASE_CSV = os.path.join(INPUT_FOLDER, "collisions_with_perception_worstcase.csv")

def compare_to_old(old_path, old_col, new_col, new_source_df, label):
    if not os.path.exists(old_path):
        print(f"{label}: no previously stored file found at {old_path}, skipping.")
        return
    old_scores = pd.read_csv(old_path, usecols=["collision_id", old_col], low_memory=False)
    check = new_source_df[["collision_id", new_col]].merge(old_scores, on="collision_id", how="inner")
    if len(check) == 0:
        print(f"{label}: no matching collision_id values found, cannot compare.")
        return
    diff = (check[new_col] - check[old_col]).abs()
    corr = check[new_col].corr(check[old_col])
    print(f"{label}: {len(check):,} collisions compared")
    print(f"  How closely they match, from 0 (no relation) to 1 (perfect): {corr:.6f}")
    print(f"  Biggest gap found between the two: {diff.max():.6f}")
    print(f"  Typical gap between the two: {diff.mean():.6f}")
    if corr > 0.999 and diff.max() < 0.01:
        print("  These match.")
    else:
        print("  These do NOT closely match.")
    print()

print("--- Average scenario ---")
compare_to_old(OLD_AVERAGE_CSV, "scenario_a_score", "perceived_safety_average", continuous_scores, "Average vs old file")

print("--- Worst case scenario ---")
compare_to_old(OLD_WORSTCASE_CSV, "scenario_b_score", "perceived_safety_worst_case", scenario_classes, "Worst case vs old file")


## Part 15: creating the three final datasets

One cleaned pre-crash file, unchanged, plus three separate files built
from it, each one adding exactly one of the three original scenarios,
so they can be compared or modelled on independently rather than all
sitting mixed together in one file.


In [ ]:
# A single lookup table with everything: the real, continuous safety
# scores AND their matching class labels, for every collision.
score_lookup = continuous_scores.merge(scenario_classes, on="collision_id", how="outer")

def build_final_dataset(score_cols, out_name, label):
    merged = precrash.copy()
    merged["collision_id"] = full["collision_id"].values
    merged = merged.merge(score_lookup[["collision_id"] + score_cols], on="collision_id", how="left")
    matched = merged[score_cols[0]].notna().sum()
    merged = merged.drop(columns=["collision_id"])
    out_path = os.path.join(INPUT_FOLDER, out_name)
    merged.to_csv(out_path, index=False)
    print(f"{label}: {matched:,} of {len(merged):,} collisions matched, saved to {out_path}")
    print(f"  Shape: {merged.shape[0]:,} rows, {merged.shape[1]} columns")
    print(f"  Safety score column(s): {score_cols}")
    return merged

# Scenario A: the real, continuous average score (0 to 2) is the safety
# score, its matching class is kept alongside for anyone who wants it too.
dataset_a = build_final_dataset(
    ["perceived_safety_average", "scenario_a_class"],
    "collisions_scenario_a_average.csv", "Scenario A (average)",
)
print()
# Scenario B: the real, continuous worst case score (0 to 1) is the safety
# score, same treatment, its matching class kept alongside.
dataset_b = build_final_dataset(
    ["perceived_safety_worst_case_score", "scenario_b_class"],
    "collisions_scenario_b_worstcase.csv", "Scenario B (worst case)",
)
print()
# Scenario C: majority vote has no continuous equivalent, the class is the
# genuine, correct safety score here.
dataset_c = build_final_dataset(
    ["scenario_c_class"],
    "collisions_scenario_c_majorityvote.csv", "Scenario C (majority vote)",
)
